# SeaBASS data to CMAP

#### Krista Longnecker, 10 June 2026
BIOS-SCOPE has some data at SeaBASS that can be prepared for CMAP. SeaBASS data download is an offline process, so use data requested for download and stored locally (e.g., not synced to GitHub).

In [1]:
%reset -f

In [2]:
import os
import pandas as pd
from datetime import date

#used this to step into the function and debug it, also need line with set_trace()
from IPython.core.debugger import set_trace
import warnings

#SB_support from NASA's website: https://seabass.gsfc.nasa.gov/wiki/readsb_python
from SB_support import readSB


In [3]:
# Suppress a specific future warning by its exact text or a Regex pattern
warnings.filterwarnings(
    "ignore", 
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes", 
    category=FutureWarning
)

In [4]:
#Check that an Excel file exists for the variable metadata:
if os.path.exists('CMAP_variableMetadata_additions.xlsx'):
    print('Found Excel file with metadata')
else:
    #You cannot proceed without the file, so stop the script if it's not found
    print(f"No Excel file with metadata found")
    sys.exit(1)
    
#will add a new worksheet with the variables from Amy's data

Found Excel file with metadata


In [5]:
#need to know what type of sample each file contains, use a function to match names
def findSampleType(oneName):
    '''Need to filter the type of sample based on the filename. We have four choices:
    # CTDauto (name will contain FlowcamBottleAuto)
    # CTDtrigger (name will contain FlowcamBottleTrigger)
    # netZooscan (name will contain ZooScanNet)
    # netFlowcam (name will contain FlowcamNet)'''
    if [item for item in [oneName] if 'FlowcamBottleAuto' in item]:
        sType = 'FlowcamBottleAuto'
    elif [item for item in [oneName] if 'FlowcamBottleTrigger' in item]:
        sType = 'FlowcamBottleTrigger'
    elif [item for item in [oneName] if 'ZooScanNet' in item]:
        sType = 'ZooScanNet'
    elif [item for item in [oneName] if 'FlowcamNet' in item]:
        sType = 'FlowcamNet'
    elif [item for item in [oneName] if '_HPLC_' in item]:
        sType = 'HPLC'
    elif [item for item in [oneName] if 'report' in item]:
        sType = 'report'
    else: 
        print('Something is wrong')
        
    return sType

In [6]:
#The math for the calculation is in the header of each file (only found this to be true for the net samples). From Amy I have 
# CTD- auto: !to calculate abundance of a particle for FlowCam Niskin Bottle Sample (Auto mode) (each row is a particle): abundance (#/L) = ((particle (unitless) / volume_imaged_mL (mL))
# CTD- trigger: !to calculate abundance of a particle for FlowCam Niskin Bottle Sample (Trigger mode) (each row is a particle): abundance (#/L) = ((particle (unitless) / volume_imaged_mL (mL))
# net - zooscan: !to calculate abundance of a particle (each row is a particle): abundance (#/L) = particle (unitless) * splitfraction (unitless) / volnet (L)
# net - flowcam: !to calculate abundance of a particle for Auto (each row is a particle): abundance (#/L) = (((particle (unitless) / volume_imaged_mL (mL)) * particle_dilution_factor (mL)) * splitfraction (unitless)) / volnet (L))
def doTheMath(df,sampleType):
        if sampleType == 'FlowcamBottleAuto':
            #set_trace()
            #from header:
            #!volume = acq_vol_img divided by 1000. This is the total volume imaged by FlowCam. 
            #!Use this volume in particle concentration calculations in AutoImage mode.
            #from Amy
            ## abundance (#/L) = ((particle (unitless) / volume_imaged_mL (mL))
            if 'volume' in df.columns:
                df['particles_perL'] = (1 / df['volume'])
            else:
                #seem to have two formats for these data
                df['particles_perL'] = -999
            
        elif sampleType == 'FlowcamBottleTrigger':
            #from header:
            #!volfilt = acq_vol_proc divided by 1000. This is the total volume pulled through FlowCam. 
            #Use this volume in particle concentration calculations in Trigger mode.
            #from Amy
            ## abundance (#/L) = ((particle (unitless) / volume_imaged_mL (mL))
            if 'volfilt' in df.columns:
                df['particles_perL'] = ((1 / df['volfilt']))
            else:
                #seem to have two formats for these data
                df['particles_perL'] = -999
                                    
        elif sampleType == 'ZooScanNet':
            #from header:
            #!volnet = volume calculated from net flowmeter in L
            #!splitfraction = the fraction of the sample that was diluted (for example if you split the sample 4 times that is 1/16th of the total sample so 16 is the value used)
            #from Amy
            ## abundance (#/L) = particle (unitless) * splitfraction (unitless) / volnet (L) #from Amy
            if 'splitfraction' in df.columns:
                df['particles_perL'] = (1 * df['splitfraction'] / df['volnet'])
            else:
                #seem to have two formats for these data
                df['particles_perL'] = -999
                                    
        elif sampleType == 'FlowcamNet':
            #from header
            #!volume_sampled_mL= FlowCam's Sample_Volume_Aspirated - total volume of sample pulled into the Flowcam system by the pump or syringe
            #!volume_imaged_mL = FlowCam's Fluid_Volume_Imaged - the exact volume of fluid within the camera's field of view and captured as images used for Auto mode
            #!volnet = volume calculated from net flowmeter in L
            #!splitfraction = the fraction of the sample that was diluted (for example if you split the sample 4 times that is 1/16th of the total sample so 16 is the value used)
            #!particle_dilution_factor (mL) = Term to encompass the conversion from particles to particles/mL - in this case the volume the net sample is diluted into before passing it through the FlowCam

            if 'volume_imaged_ml' in df.columns:
                ## abundance (#/L) = (((particle (unitless) / volume_imaged_mL (mL)) * particle_dilution_factor (mL)) * splitfraction (unitless)) / volnet (L))
                #as received is missing one "(" at the beginning
                df['particles_perL'] = ((((1 / df['volume_imaged_ml']) * df['particle_dilution_factor']) * df['splitfraction']) / df['volnet'])
            else:
                #seem to have two formats for these data
                df['particles_perL'] = -999
                #set_trace()
            
        elif sampleType == 'HPLC':
            #do nothing
            df['particles_perL'] = pd.NA
                                    
        elif sampleType == 'report':
            #do nothing
            df['particles_perL'] = pd.NA
                                    
            
        return df

In [7]:
#go find the data in the data folder (it is in .gitignore, so it will not end up at GitHub). Makes a list
# I know the folder is there, so no need to make the folder
folder = "data"
SBstring = ".sb"
matching_files = []

os.chdir(".")

for root, dirs, files in os.walk(folder):
    for file in files:
        full_path = os.path.join(root, file)
        # Change 'file' to 'full_path' if you want to search the entire folder path
        if SBstring in file:  
            matching_files.append(full_path)

In [8]:
#get the list of unique cruises from the folder structure (need this for the CMAP submission)

# Target directory path
folder_path = 'data/SeaBASS/Maas/PVST_BATS_PLANKTON'

# Filter for folders only (requires joining path to verify accurately)
unCruises = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]

In [9]:
#first up, read in the SeaBASS file and make a square matrix out of all the data sets. 
#I am going to group all cruises into one matrix

#read in the first file and use that to start the dataframe 
#otherwise I end up with rows grouped by cruise and I am not sure how to get around that
idx = 0
one = readSB(filename = matching_files[idx])
# #use the SeaBASS experiment name that will be used for the Excel file name
# dataset_short_name = one.headers['experiment'] 

#then process the data needed to start the data frame
oneD = one.data
df = pd.DataFrame(oneD)

#add a columm with filename while I sort out the issues
df['filename'] = one.filename

#add a column with sample type so I know what calculation to do later
st = findSampleType(one.filename)
df['sampleType'] = st

#do the math to get to particles per L, need the function as the calculation varies for each sample type
df = doTheMath(df,st)

In [10]:
#then go through and add the remaining files...start loop at 2
for idx in range(2,(len(matching_files))):
    one = readSB(filename = matching_files[idx],no_warn=True)
    oneD = one.data
    temp = pd.DataFrame(oneD)
    temp['filename'] = one.filename
    
    temp['sampleType'] = findSampleType(one.filename)
    st = findSampleType(one.filename)
    temp['sampleType'] = st
    temp = doTheMath(temp,st)

    if 'associatedmedia' in one.variables: # data file with details on zoop
        #print('work on this')

        #remove URL from associatedmedia and leave the object details (otherwise have issues in export)
        temp['associatedmedia'] = temp['associatedmedia'].str.strip('https://ecotaxa.obs-vlfr.fr/objectdetails/')
        temp = temp.rename(columns = {'associatedmedia': "objectdetails"})

        #get lat, lon
        temp['lat'] = (one.headers['north_latitude']).replace('[DEG]','') # was: 32.190[DEG]
        temp['lon'] = (one.headers['east_longitude']).replace('[DEG]','') # was: -64.532[DEG]  

        #then time
        time_str = one.headers['start_date'] + " " + (one.headers['start_time']).replace('[GMT]','')
        temp['ts'] = pd.to_datetime(time_str,format='%Y%m%d %H:%M:%S')             
        #now that I have added what is required, pop out of the if statement as the next steps are the same for both
    else:
        #print('normal datafile')
        #may as well convert time here as well (already have lat / lon)
        temp['ts'] = pd.to_datetime(temp['date'].astype(str) + ' ' + temp['time'].astype(str))
        #now that I have added what is required, pop out of the if statement as the next steps are the same for both

    #temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "+00:00Z")
    temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "Z")
    #now delete the temporary column 
    temp = temp.drop(columns = 'ts')                      
    df = pd.concat([df,temp],ignore_index=True)

In [11]:
df.shape

(225448, 73)

In [28]:
#reorder to make it easier to figure out wtf is going on
target_col = "filename"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "sampleType"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "particles_perL"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]


target_col = "volfilt"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]


target_col = "volume_sampled_ml"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]


target_col = "volume_imaged_ml"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "volnet"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "splitfraction"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "particle_dilution_factor"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "biovolume"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]




In [ ]:
df.to_excel('data/stepR.xlsx')

In [16]:
#need to figure out what the column names are for the cases where I ended up with -999
dft = df[df['particles_perL'] ==-999]

In [20]:
one = dft[dft['sampleType']=='FlowcamNet']

In [21]:
one

,hplc_gsfc_id,station,depth,water_depth,bottle,date,time,lat,lon,volfilt,...,area_cross_section,length_representation,width_representation,equivalent_spherical_diameter,volume_sampled_ml,volume_imaged_ml,volnet,splitfraction,particle_dilution_factor,biovolume
59543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,62.06,12.02,8.38,10.51,NaN,NaN,NaN,NaN,NaN,NaN
59544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,16.98,17.12,4.01,11.94,NaN,NaN,NaN,NaN,NaN,NaN
59545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,65.17,12.75,8.38,10.53,NaN,NaN,NaN,NaN,NaN,NaN
59546,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,127.24,16.40,11.29,14.19,NaN,NaN,NaN,NaN,NaN,NaN
59547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,786.49,49.92,30.24,39.39,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.667,-64.164,0.003517,...,1860.91,60.68,42.57,53.29,NaN,NaN,NaN,NaN,NaN,NaN
224023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.667,-64.164,0.003517,...,312.54,22.64,19.02,21.28,NaN,NaN,NaN,NaN,NaN,NaN
224024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.667,-64.164,0.003517,...,654.44,55.25,42.57,49.16,NaN,NaN,NaN,NaN,NaN,NaN
224025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.667,-64.164,0.003517,...,1421.09,48.00,42.57,44.68,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
udf = one.drop_duplicates(subset = ['filename'])

In [26]:
udf

,hplc_gsfc_id,station,depth,water_depth,bottle,date,time,lat,lon,volfilt,...,area_cross_section,length_representation,width_representation,equivalent_spherical_diameter,volume_sampled_ml,volume_imaged_ml,volnet,splitfraction,particle_dilution_factor,biovolume
59543,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.000325,...,62.06,12.02,8.38,10.51,NaN,NaN,NaN,NaN,NaN,NaN
61434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.017853,...,7245.47,100.66,93.59,98.01,NaN,NaN,NaN,NaN,NaN,NaN
63413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.661,-64.156,0.002219,...,8358.85,105.97,102.34,104.46,NaN,NaN,NaN,NaN,NaN,NaN
92370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.673,-64.164,0.000315,...,36.94,15.67,6.92,12.21,NaN,NaN,NaN,NaN,NaN,NaN
94323,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.673,-64.164,0.014604,...,5959.37,111.25,72.40,95.65,NaN,NaN,NaN,NaN,NaN,NaN
96248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.673,-64.164,0.001936,...,1587.04,75.17,38.95,60.23,NaN,NaN,NaN,NaN,NaN,NaN
168190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.676,-64.196,0.000053,...,10.18,15.67,5.47,11.36,NaN,NaN,NaN,NaN,NaN,NaN
170190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.676,-64.196,0.002012,...,1056.43,58.27,33.55,43.85,NaN,NaN,NaN,NaN,NaN,NaN
172190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.676,-64.196,0.000462,...,395.48,31.70,20.83,25.91,NaN,NaN,NaN,NaN,NaN,NaN
189996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.665,-64.035,0.000143,...,112.97,17.85,13.48,15.97,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
udf.to_excel('data/un.xlsx')

In [ ]:
#now get ready to put this into CMAP format
# Required variables are time, lat, lon, depth #df = pd.DataFrame(columns=['time','lat','lon','depth'])

target_col = "depth"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "lon"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "lat"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]

target_col = "date_cmap"
new_order = [target_col] + [col for col in df.columns if col != target_col]
df = df[new_order]


df = df.rename(columns = {'date_cmap': "time"})

In [ ]:
'''
Work on the second sheet: metadata about the variables; use the CMAP dataset template to setup the dataframe 
so I get the column headers right
'''
fName = 'datasetTemplate.xlsx' #downloaded from CMAP website
sheet_name = 'vars_meta_data'
vars = pd.read_excel(fName, sheet_name=sheet_name)
cols = vars.columns.tolist()
#df2 will be the dataframe with the metadata about the variables, set it up empty here
nVariables = df.shape[1]
df2 = pd.DataFrame(columns=cols,index = pd.RangeIndex(0,nVariables,1))

# this is only a partial list of variables for the moment
df2['var_short_name'] = df.columns
df2.loc[:,'var_long_name'] = ''
df2.loc[:,'var_sensor'] = ''
df2.loc[:,'var_unit'] = ''
df2.loc[:,('var_spatial_res')] = 'irregular'
df2.loc[:, ('var_temporal_res')] = 'irregular'
df2.loc[:,('var_discipline')] = 'biology'
df2.loc[:,('visualize')] = 0 #many would be hard to visualize these one at a time: edit later

#add the keywords, nothing fancy here
misc_keywords = 'BIOS-SCOPE, BIOS, Simons Foundation International, bio, biogeo, biogeochemistry, biology, flowCAM, Zooscan, net tows, Discrete, Bottle, cruise, Discrete, in situ, insitu, in-situ, North Atlantic Ocean, observation'
df2.loc[:,('var_keywords')] = misc_keywords
                         

In [ ]:
#uncomment this out the first time to get the variable information, but then once the information is in Excel, read 
#the information that has been copied into fName = 'CMAP_variableMetadata_additions.xlsx'
#there is most certainly a better way to do this, but I am not going to deal with that now

# #SeaBASS does not appear to have a downloadable file with their variable names, though the information is here:
# #https://seabass.gsfc.nasa.gov/wiki/stdfields
# #copy/paste into Excel, tidy up, and use to get the details I need on units and long names
# fName = 'SeaBASS_variableNames.xlsx'
# vn = pd.read_excel(fName,sheet_name = 'variables')
# vn['var_short_name'] = vn['Field name'].str.lower() #otherwise I miss some of the matches :(

# df2 = pd.merge(df2,vn,how='left')
# df2['var_long_name'] = df2['Description']
# df2['var_unit'] = df2['Units']
# df2 = df2.drop(columns = ['Description','Units','Field name'])

In [ ]:
#there are a few pieces of metadata that CMAP wants that will be easier to track in an Excel file -
#make the file once, and then update as needed for future BCO-DMO datasets.
#The keywords include cruises, and all possible names for a variable. I wonder if
#CMAP has that information available in a way that can be searched?
#These are all one-offs...so it's rather hard to automate. I am coming to the idea it will be easier to manually annotate the files.
#Come back to this later.

# # Note that I made the Excel file after I started down this rabbit hole with the sensors. It will probably make sense
# #to pull the sensor information from the file as well.
fName = 'CMAP_variableMetadata_additions.xlsx'
sheetName = exportFile[0:31] #Excel limits the length of the sheet name
moreMD = pd.read_excel(fName,sheet_name = sheetName)

# #suffixes are added to column name to keep them separate; '' adds nothing while '_td' adds _td that can get deleted next
df2 = moreMD.merge(df2[['var_short_name','var_keywords']],on='var_short_name',how='right',suffixes=('', '_td',))
# # Discard the columns that acquired a suffix:
df2 = df2[[c for c in df2.columns if not c.endswith('_td')]]

In [ ]:
'''
Gather the details I need about the dataset
'''
project_description = 'The data provided by this project is the abundance and taxonomic information for the 5-300 µm planktonic community captured via imaging with a FlowCam or Zooscan. We additionally analyze a duplicate subsetÂ of water samples from the same cast as BATS HPLC depths (0-250 m), sending these pigment samples to GSFC. Target depths for both the niskin-based FlowCam and HPLC samples include surface, 40, 80, 120, 160, and 200. The FlowCam was run in both trigger and automated mode at 10X for each bottle sample to enumerate and classify photosynthetic and non-photosynthetic organisms. In addition, the standard Bermuda Atlantic Time Series Site phytoplankton net tows (0-175 mwo, 120-150 m depth, 35 µm mesh net) were imaged. Images were taken using automatic mode at 2x, 4x, and 10x with the FlowCam, while two size fractions (>1000, <1000) were captured with a Zooscan to ensure enumeration of the larger phytoplankton taxonomic groups.'

# gather up the dataset_meta_data into df3

df3 = pd.DataFrame({
    'dataset_short_name': ['BS_PLANKTON_1'],
    'dataset_long_name': ['BS_PVST_BATS_PLANKTON'], 
    'dataset_version': ['1.0'],
    'dataset_release_date': [date.today()],
    'dataset_make': ['observation'],
    'dataset_source': ['Amy Maas, Bermuda Institute of Ocean Sciences - Arizona State University'],
    'dataset_distributor': ['Amy Maas, Bermuda Institute of Ocean Sciences - Arizona State University'],
    'dataset_acknowledgement': ['This work supported by funding from NASA.'],
    'dataset_history': [''],
    'dataset_description': [project_description],
    'dataset_references': ['DOI is 10.5067/SeaBASS/PVST_BATS_PLANKTON/DATA001; '],
    'climatology': [0]
    })

In [ ]:
#insert the list of unique cruises from the data pulled from the folder structure
df3 = pd.concat([df3,pd.DataFrame(unCruises)],axis=1)
df3 = df3.rename(columns={df3.columns[-1]: 'cruise_names'})

In [ ]:
fName_CMAP = 'data/' + 'BIOS-SCOPE_SeaBASS_' + 'PVST_BATS_PLANKTON' + '.xlsx'
dataset_names = {'data': df, 'dataset_meta_data': df3, 'vars_meta_data': df2}
with pd.ExcelWriter(fName_CMAP) as writer:
    for sheet_name, data in dataset_names.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
# uncomment out the next line if I want to put code that will not be executed below this point
raise SystemExit("Stop execution here")

In [ ]:
data.headers

In [ ]:
data.length

In [ ]:
data.bdl

In [ ]:
data.filename

In [ ]:
data.pi

In [ ]:
data.headers

In [ ]:
data.variables

In [ ]:
#then go through and add the remaining files...start loop at 2

for idx in range(2,(len(matching_files))):
    one = readSB(filename = matching_files[idx], no_warn=True)
    oneD = one.data
    temp = pd.DataFrame(oneD)
    d = [col for col in temp.columns if 'volume' in col]
    if d:
        print(one.filename)
    temp['filename'] = one.filename
    df_working = pd.concat([df_working,temp],ignore_index=True)

In [ ]:
# need to trap the files with the image information because they do not ahve the headers with the station information
# Add that in (and remove the URLs as they cannot be exported as an Excel file); will pull the objid from the URL
# in order to retain the information

idx = 4
one = readSB(filename = matching_files[idx],no_warn=True)
oneD = one.data
temp = pd.DataFrame(oneD)
temp['filename'] = one.filename

if 'associatedmedia' in one.variables: # data file with details on zoop
    #print('work on this')
    
    #remove URL from associatedmedia and leave the object details (otherwise have issues in export)
    temp['associatedmedia'] = temp['associatedmedia'].str.strip('https://ecotaxa.obs-vlfr.fr/objectdetails/')
    temp = temp.rename(columns = {'associatedmedia': "objectdetails"})
        
    #get lat, lon
    temp['lat'] = (one.headers['north_latitude']).replace('[DEG]','') # was: 32.190[DEG]
    temp['lon'] = (one.headers['east_longitude']).replace('[DEG]','') # was: -64.532[DEG]  
    
    #then time
    time_str = one.headers['start_date'] + " " + (one.headers['start_time']).replace('[GMT]','')
    temp['ts'] = pd.to_datetime(time_str,format='%Y%m%d %H:%M:%S')             
    #now that I have added what is required, pop out of the if statement as the next steps are the same for both
else:
    #print('normal datafile')
    #may as well convert time here as well (already have lat / lon)
    temp['ts'] = pd.to_datetime(temp['date'].astype(str) + ' ' + temp['time'].astype(str))
    #now that I have added what is required, pop out of the if statement as the next steps are the same for both
                   
#temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "+00:00Z")
temp['date_cmap'] = temp['ts'].dt.strftime("%Y-%m-%dT%H:%M:%S" + "Z")
#now delete the temporary column 
temp = temp.drop(columns = 'ts')                      
df_working = pd.concat([df_working,temp],ignore_index=True)

In [ ]:
#then go through and add the remaining files...start loop at 2
for idx in range((len(matching_files))):
    one = readSB(filename = matching_files[idx], no_warn=True)
    oneD = one.data
    df_working = pd.DataFrame(oneD)
    d = [col for col in dfW.columns if 'volume' in col]
    if d:
        print(one.filename)

In [ ]:
filt = [item for item in matching_files if '_HPLC_' not in item]
filt[:] = [item for item in filt if 'report' not in item]

#check that I find all that I need (filt should be empty at the end of this)
filt[:] = [item for item in filt if 'FlowcamBottleAuto' not in item]
filt[:] = [item for item in filt if 'FlowcamBottleTrigger' not in item]
filt[:] = [item for item in filt if 'ZooScanNet' not in item]
filt[:] = [item for item in filt if 'FlowcamNet' not in item]

if not filt:
    print("This should be true (e.g., found all file types)")
else:
    print("Something is wrong there are still files on this list")


In [ ]:
idx = 23
matching_files[23]


one = readSB(filename = matching_files[idx])
# #use the SeaBASS experiment name that will be used for the Excel file name
# dataset_short_name = one.headers['experiment'] 

#then process the data needed to start the data frame
oneD = one.data
df = pd.DataFrame(oneD)

#add a columm with filename while I sort out the issues
df['filename'] = one.filename

#add a column with sample type so I know what calculation to do later
df['sampleType'] = findSampleType(one.filename)